<div dir="rtl">

# جلسه ۴: حلقه آموزش استاندارد: nn.Module، DataLoader و Fashion-MNIST

**The Standard Training Loop: nn.Module, DataLoader and Fashion-MNIST**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/keyvanmahmoudi/Deep_Learning_Course/blob/main/04.%20Training_Loop_FashionMNIST.ipynb)

**دوره یادگیری عمیق و شبکه‌های عصبی** — کیوان محمودی

### اهداف این جلسه

- بارگذاری و پیش‌پردازش داده تصویری با `torchvision` و `transforms`
- درک `Dataset` و `DataLoader` و ساخت یک Dataset سفارشی
- تعریف مدل با ارث‌بری از `nn.Module` و `nn.Sequential`
- نوشتن توابع قابل استفاده مجدد `train_one_epoch` و `evaluate` با `model.train()` و `model.eval()`
- ارزیابی با Confusion Matrix، ذخیره و بارگذاری مدل و پیش‌بینی روی یک نمونه

> **نکته اجرا:** این نوتبوک روی GPU سریع‌تر اجرا می‌شود. در Colab از منوی `Runtime → Change runtime type` گزینه **T4 GPU** را انتخاب کنید.

</div>

<div dir="rtl">

## ۱. مسئله: طبقه‌بندی تصاویر لباس

**Fashion-MNIST** (Xiao et al., 2017) شامل ۷۰٬۰۰۰ تصویر خاکستری ۲۸×۲۸ پیکسلی از ۱۰ نوع لباس و کفش است: ۶۰٬۰۰۰ تصویر آموزشی و ۱۰٬۰۰۰ تصویر آزمون. این دیتاست جایگزین سخت‌تری برای MNIST (ارقام دست‌نویس) است و برای آزمودن سریع مدل‌ها بسیار مناسب است.

در این جلسه همه اجزای یک **پروژه استاندارد PyTorch** را کنار هم می‌گذاریم. ساختاری که اینجا می‌سازیم تا آخر دوره، با تغییرات کوچک، تکرار می‌شود:

۱. داده: `Dataset` ← `DataLoader`

۲. مدل: کلاسی که از `nn.Module` ارث می‌برد

۳. تابع هزینه و Optimizer

۴. حلقه آموزش و ارزیابی

۵. تحلیل نتایج و ذخیره مدل

</div>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import datasets, transforms

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

<div dir="rtl">

## ۲. بارگذاری داده با torchvision

ماژول `torchvision.datasets` دیتاست‌های معروف بینایی ماشین را دانلود و آماده می‌کند. با آرگومان `transform` مشخص می‌کنیم هر تصویر قبل از تحویل چه پیش‌پردازشی ببیند:

- `ToTensor()`: تصویر PIL با مقادیر صفر تا ۲۵۵ را به تنسور `float32` با شکل `(C, H, W)` و مقادیر صفر تا یک تبدیل می‌کند.
- `Normalize(mean, std)`: هر کانال را استاندارد می‌کند: $x' = (x - \mu)/\sigma$. ورودی‌های با میانگین نزدیک صفر و واریانس نزدیک یک، آموزش را پایدارتر و سریع‌تر می‌کنند (همان Feature Scaling در دوره یادگیری ماشین).

میانگین و انحراف معیار را **فقط از داده آموزشی** حساب می‌کنیم تا اطلاعات داده آزمون به مدل نشت نکند (Data Leakage).

</div>

In [ ]:
raw_train = datasets.FashionMNIST(root='data', train=True, download=True, transform=transforms.ToTensor())
pixels = raw_train.data.float() / 255.0
MEAN, STD = pixels.mean().item(), pixels.std().item()
print(f'training-set mean = {MEAN:.4f}, std = {STD:.4f}')

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((MEAN,), (STD,)),
])

full_train = datasets.FashionMNIST(root='data', train=True, download=True, transform=transform)
test_set = datasets.FashionMNIST(root='data', train=False, download=True, transform=transform)
CLASSES = full_train.classes
print('classes:', CLASSES)
print('train size:', len(full_train), ' test size:', len(test_set))

In [ ]:
image, label = full_train[0]
print('one sample ->', image.shape, image.dtype, '| label:', label, CLASSES[label])

fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for ax, idx in zip(axes.flat, np.random.choice(len(raw_train), 16, replace=False)):
    img, lab = raw_train[idx]
    ax.imshow(img.squeeze(), cmap='gray')
    ax.set_title(CLASSES[lab], fontsize=9)
    ax.axis('off')
plt.tight_layout(); plt.show()

<div dir="rtl">

### داده اعتبارسنجی (Validation Set)

برای انتخاب ابرپارامترها (تعداد لایه، نرخ یادگیری، تعداد Epoch و …) **نباید** از داده آزمون استفاده کنیم؛ وگرنه ارزیابی نهایی خوش‌بینانه و غیرواقعی می‌شود. پس ۱۰٬۰۰۰ تصویر از داده آموزشی را به عنوان **داده اعتبارسنجی (Validation)** جدا می‌کنیم:

| بخش | کاربرد |
|---|---|
| Train | به‌روزرسانی وزن‌ها |
| Validation | انتخاب ابرپارامتر و پایش Overfitting در طول آموزش |
| Test | فقط یک بار، برای گزارش نهایی |

</div>

In [ ]:
train_set, val_set = random_split(full_train, [50_000, 10_000], generator=torch.Generator().manual_seed(SEED))
print('train:', len(train_set), ' validation:', len(val_set), ' test:', len(test_set))

<div dir="rtl">

## ۳. Dataset و DataLoader

**`Dataset`** یک رابط ساده است: فقط باید دو متد داشته باشد:

- `__len__`: تعداد نمونه‌ها
- `__getitem__(i)`: نمونه $i$ام به صورت `(ورودی، برچسب)`

**`DataLoader`** روی یک Dataset، دسته‌بندی (Batching)، به‌هم‌ریختن ترتیب (Shuffling) و بارگذاری موازی (با `num_workers`) را انجام می‌دهد. برای درک بهتر، یک Dataset سفارشی کوچک از تنسورها می‌سازیم؛ در پروژه‌های واقعی (مثلاً خواندن فایل‌های تصویر از دیسک یا متن از CSV) همین الگو را استفاده می‌کنید:

</div>

In [ ]:
class TensorPairDataset(Dataset):
    """Minimal custom dataset: wraps an input tensor and a label tensor."""

    def __init__(self, X, y):
        assert len(X) == len(y)
        self.X, self.y = X, y

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


toy = TensorPairDataset(torch.randn(100, 3), torch.randint(0, 2, (100,)))
toy_loader = DataLoader(toy, batch_size=32, shuffle=True)
for xb, yb in toy_loader:
    print('batch:', tuple(xb.shape), tuple(yb.shape))

<div dir="rtl">

دقت کنید که دسته آخر فقط ۴ نمونه دارد ($100 = 3\times32 + 4$). اگر لازم باشد همه دسته‌ها هم‌اندازه باشند از `drop_last=True` استفاده می‌کنیم.

حالا DataLoaderهای Fashion-MNIST را می‌سازیم. داده آموزشی را Shuffle می‌کنیم تا ترتیب نمونه‌ها در هر Epoch تصادفی باشد؛ برای ارزیابی نیازی به Shuffle نیست.

</div>

In [ ]:
BATCH_SIZE = 128
NUM_WORKERS = 2

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader = DataLoader(val_set, batch_size=512, shuffle=False, num_workers=NUM_WORKERS)
test_loader = DataLoader(test_set, batch_size=512, shuffle=False, num_workers=NUM_WORKERS)

xb, yb = next(iter(train_loader))
print('images:', tuple(xb.shape), xb.dtype, '| labels:', tuple(yb.shape), yb.dtype)
print('batches per epoch:', len(train_loader))

<div dir="rtl">

## ۴. تعریف مدل با nn.Module

هر مدل در PyTorch کلاسی است که از `nn.Module` ارث می‌برد:

- در `__init__` لایه‌ها را به عنوان ویژگی (Attribute) تعریف می‌کنیم. PyTorch پارامترهای آن‌ها را خودکار ثبت می‌کند و `model.parameters()` همه را برمی‌گرداند.
- در `forward` مسیر عبور داده را می‌نویسیم. هیچ‌وقت `forward` را مستقیم صدا نمی‌زنیم؛ `model(x)` آن را (به‌همراه Hookهای داخلی) اجرا می‌کند.

**نکته مهم:** لایه آخر **Softmax ندارد** و خروجی‌اش **Logits** است. تابع `nn.CrossEntropyLoss` خودش Softmax و لگاریتم را به شکلی پایدار از نظر عددی (Log-Sum-Exp) ترکیب می‌کند. اضافه کردن Softmax در مدل یک اشتباه رایج است که آموزش را کند و ناپایدار می‌کند.

</div>

In [ ]:
class MLP(nn.Module):
    def __init__(self, in_features=28 * 28, hidden=(256, 128), n_classes=10):
        super().__init__()
        self.flatten = nn.Flatten()                      # (N, 1, 28, 28) -> (N, 784)
        self.fc1 = nn.Linear(in_features, hidden[0])
        self.fc2 = nn.Linear(hidden[0], hidden[1])
        self.out = nn.Linear(hidden[1], n_classes)
        self.act = nn.ReLU()

    def forward(self, x):
        x = self.flatten(x)
        x = self.act(self.fc1(x))
        x = self.act(self.fc2(x))
        return self.out(x)                               # logits: no softmax here


def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


model = MLP().to(device)
print(model)
print(f'trainable parameters: {count_parameters(model):,}')

<div dir="rtl">

برای مدل‌های ساده و ترتیبی، `nn.Sequential` همین کار را کوتاه‌تر انجام می‌دهد. هر دو تعریف معادل‌اند؛ اما برای مدل‌هایی با مسیرهای انشعابی یا منطق شرطی (مثل ResNet در جلسه ۸) کلاس سفارشی لازم است:

</div>

In [ ]:
sequential_model = nn.Sequential(
    nn.Flatten(),
    nn.Linear(784, 256), nn.ReLU(),
    nn.Linear(256, 128), nn.ReLU(),
    nn.Linear(128, 10),
)
print('same number of parameters:', count_parameters(sequential_model) == count_parameters(model))

with torch.no_grad():
    logits = model(xb.to(device))
print('logits shape:', tuple(logits.shape))
print('probabilities of the first sample:', torch.softmax(logits[0], dim=0).cpu().numpy().round(3))

<div dir="rtl">

مدل آموزش‌ندیده تقریباً به همه کلاس‌ها احتمال برابر (حدود $0.1$) می‌دهد. Loss اولیه هم باید نزدیک $-\log(0.1) \approx 2.30$ باشد؛ این یک **بررسی سلامت (Sanity Check)** مفید قبل از شروع آموزش است.

## ۵. توابع آموزش و ارزیابی

دو تابع می‌نویسیم که در همه جلسات بعد دوباره استفاده می‌شوند. به دو حالت مدل دقت کنید:

- `model.train()`: حالت آموزش. لایه‌هایی مثل Dropout و BatchNorm (جلسه ۶) در این حالت رفتار آموزشی دارند.
- `model.eval()`: حالت ارزیابی. همراه با `torch.no_grad()` برای صرفه‌جویی در حافظه و زمان.

حتی اگر مدل فعلی ما این لایه‌ها را ندارد، عادت درست این است که همیشه حالت را تنظیم کنیم.

</div>

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, correct, n = 0.0, 0, 0
    for X, y in loader:
        X, y = X.to(device), y.to(device)
        logits = model(X)                    # 1. forward
        loss = criterion(logits, y)          # 2. loss
        optimizer.zero_grad()                # 3. reset gradients
        loss.backward()                      # 4. backward
        optimizer.step()                     # 5. update
        total_loss += loss.item() * len(y)
        correct += (logits.argmax(dim=1) == y).sum().item()
        n += len(y)
    return total_loss / n, correct / n


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, n = 0.0, 0, 0
    for X, y in loader:
        X, y = X.to(device), y.to(device)
        logits = model(X)
        total_loss += criterion(logits, y).item() * len(y)
        correct += (logits.argmax(dim=1) == y).sum().item()
        n += len(y)
    return total_loss / n, correct / n


criterion = nn.CrossEntropyLoss()
print(f'initial validation loss: {evaluate(model, val_loader, criterion, device)[0]:.3f}  (expected ~ {np.log(10):.3f})')

<div dir="rtl">

## ۶. آموزش مدل

از `SGD` با Momentum استفاده می‌کنیم (جزئیات Momentum و Adam در جلسه ۵). در پایان هر Epoch، Loss و دقت را روی داده آموزش و اعتبارسنجی ثبت می‌کنیم.

</div>

In [ ]:
EPOCHS = 15
LR = 0.05

model = MLP().to(device)
optimizer = torch.optim.SGD(model.parameters(), lr=LR, momentum=0.9)
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    va_loss, va_acc = evaluate(model, val_loader, criterion, device)
    for k, v in zip(history, (tr_loss, tr_acc, va_loss, va_acc)):
        history[k].append(v)
    print(f'epoch {epoch:2d} | train loss {tr_loss:.4f} acc {tr_acc:.4f} | val loss {va_loss:.4f} acc {va_acc:.4f}')

In [ ]:
def plot_history(history):
    epochs = range(1, len(history['train_loss']) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    axes[0].plot(epochs, history['train_loss'], 'o-', label='train')
    axes[0].plot(epochs, history['val_loss'], 's-', label='validation')
    axes[0].set_title('loss'); axes[0].set_xlabel('epoch'); axes[0].legend()
    axes[1].plot(epochs, history['train_acc'], 'o-', label='train')
    axes[1].plot(epochs, history['val_acc'], 's-', label='validation')
    axes[1].set_title('accuracy'); axes[1].set_xlabel('epoch'); axes[1].legend()
    plt.show()


plot_history(history)

<div dir="rtl">

**تحلیل نمودار:** معمولاً Loss آموزش پیوسته کم می‌شود، اما Loss اعتبارسنجی بعد از چند Epoch ثابت می‌ماند یا حتی زیاد می‌شود. فاصله بین این دو منحنی نشانه **بیش‌برازش (Overfitting)** است: مدل جزئیات و نویز داده آموزشی را حفظ می‌کند. ابزارهای مقابله با Overfitting موضوع جلسه ۶ است.

## ۷. ارزیابی نهایی روی داده آزمون

حالا، و فقط یک بار، مدل را روی داده آزمون ارزیابی می‌کنیم:

</div>

In [ ]:
test_loss, test_acc = evaluate(model, test_loader, criterion, device)
print(f'test loss {test_loss:.4f} | test accuracy {test_acc:.4f}')

<div dir="rtl">

### ماتریس درهم‌ریختگی (Confusion Matrix)

دقت کلی همه چیز را نشان نمی‌دهد. Confusion Matrix نشان می‌دهد مدل کدام کلاس‌ها را با هم اشتباه می‌گیرد:

</div>

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report


@torch.no_grad()
def predict_all(model, loader, device):
    model.eval()
    preds, labels = [], []
    for X, y in loader:
        preds.append(model(X.to(device)).argmax(dim=1).cpu())
        labels.append(y)
    return torch.cat(preds).numpy(), torch.cat(labels).numpy()


y_pred, y_true = predict_all(model, test_loader, device)
fig, ax = plt.subplots(figsize=(9, 8))
ConfusionMatrixDisplay(confusion_matrix(y_true, y_pred), display_labels=CLASSES).plot(ax=ax, cmap='Blues', xticks_rotation=45, colorbar=False)
plt.title('Confusion matrix (test set)'); plt.show()

print(classification_report(y_true, y_pred, target_names=CLASSES, digits=3))

<div dir="rtl">

بیشترین اشتباه معمولاً بین کلاس‌های **Shirt**، **T-shirt/top**، **Pullover** و **Coat** است که حتی برای انسان در تصاویر ۲۸×۲۸ شبیه به هم هستند. یک MLP تصویر را به یک بردار ۷۸۴تایی تبدیل می‌کند و **ساختار فضایی (Spatial Structure)** پیکسل‌ها را نادیده می‌گیرد. در جلسه ۷ با **شبکه‌های کانولوشنی (CNN)** این مشکل را حل می‌کنیم.

نمونه‌هایی که مدل اشتباه طبقه‌بندی کرده است:

</div>

In [ ]:
wrong = np.where(y_pred != y_true)[0]
fig, axes = plt.subplots(2, 6, figsize=(14, 5))
for ax, idx in zip(axes.flat, np.random.choice(wrong, 12, replace=False)):
    img = test_set.data[idx]
    ax.imshow(img, cmap='gray')
    ax.set_title(f'true: {CLASSES[y_true[idx]]}\npred: {CLASSES[y_pred[idx]]}', fontsize=9, color='crimson')
    ax.axis('off')
plt.tight_layout(); plt.show()

<div dir="rtl">

## ۸. ذخیره و بارگذاری مدل

روش توصیه‌شده ذخیره **`state_dict`** است: یک دیکشنری از نام پارامترها به تنسورهای آن‌ها. برای بارگذاری، ابتدا یک نمونه از همان کلاس مدل می‌سازیم و سپس وزن‌ها را در آن بار می‌کنیم.

</div>

In [ ]:
torch.save(model.state_dict(), 'fashion_mlp.pt')

loaded = MLP().to(device)
loaded.load_state_dict(torch.load('fashion_mlp.pt', map_location=device))
print('loaded model test accuracy:', round(evaluate(loaded, test_loader, criterion, device)[1], 4))
print('state_dict keys:', list(loaded.state_dict().keys()))

<div dir="rtl">

## ۹. پیش‌بینی روی یک نمونه

هنگام استفاده از مدل در عمل (Inference) سه نکته را فراموش نکنید: `model.eval()`، `torch.no_grad()` و افزودن **بعد Batch** با `unsqueeze(0)`، چون مدل همیشه یک دسته از ورودی‌ها را انتظار دارد.

</div>

In [ ]:
@torch.no_grad()
def predict_image(model, image):
    model.eval()
    logits = model(image.unsqueeze(0).to(device))         # (1, 1, 28, 28) -> (1, 10)
    return torch.softmax(logits, dim=1).squeeze(0).cpu()


idx = 7
image, label = test_set[idx]
probs = predict_image(loaded, image)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5), gridspec_kw={'width_ratios': [1, 2.5]})
axes[0].imshow(test_set.data[idx], cmap='gray'); axes[0].axis('off'); axes[0].set_title(f'true: {CLASSES[label]}')
axes[1].barh(CLASSES, probs.numpy()); axes[1].invert_yaxis(); axes[1].set_xlim(0, 1)
axes[1].set_title(f'prediction: {CLASSES[probs.argmax()]} ({probs.max():.1%})')
plt.tight_layout(); plt.show()

<div dir="rtl">

## ۱۰. جمع‌بندی

- `torchvision.datasets` و `transforms` داده تصویری را آماده و نرمال می‌کنند؛ آمار نرمال‌سازی فقط از داده آموزشی.
- داده را به سه بخش Train، Validation و Test تقسیم می‌کنیم و Test فقط یک بار استفاده می‌شود.
- `Dataset` فقط `__len__` و `__getitem__` لازم دارد؛ `DataLoader` دسته‌بندی و Shuffle را انجام می‌دهد.
- مدل = کلاسی از `nn.Module` با `__init__` و `forward`؛ خروجی Logits است و Softmax داخل `CrossEntropyLoss` است.
- `model.train()` در آموزش، و `model.eval()` همراه `torch.no_grad()` در ارزیابی.
- `state_dict` روش استاندارد ذخیره مدل است.
- MLP به حدود ۸۸٪ دقت روی داده آزمون می‌رسد اما ساختار فضایی تصویر را نادیده می‌گیرد؛ CNN در جلسه ۷.

## ۱۱. تمرین

۱. تعداد نورون‌های لایه‌های پنهان را به `(512, 256)` و سپس `(64, 32)` تغییر دهید. اثر آن روی تعداد پارامتر، دقت اعتبارسنجی و فاصله منحنی‌های Train و Validation را گزارش کنید.

۲. نرمال‌سازی (`Normalize`) را حذف کنید و دوباره آموزش دهید. سرعت همگرایی چه تغییری می‌کند؟

۳. یک `Dataset` سفارشی بنویسید که داده دوره یادگیری ماشین (مثلاً `Churn_Modelling.csv`) را از CSV بخواند و یک MLP برای پیش‌بینی خروج مشتری آموزش دهید.

۴. تابع `train_one_epoch` را طوری تغییر دهید که بعد از هر ۱۰۰ Batch، میانگین Loss همان ۱۰۰ Batch را چاپ کند.

۵. حلقه آموزش را طوری تغییر دهید که بهترین مدل (بیشترین دقت اعتبارسنجی) را در طول آموزش ذخیره کند و در پایان همان را روی داده آزمون ارزیابی کنید.

</div>